# Phase 2 — Verifikasi preprocessing yang sudah tersedia
Notebook membaca hasil lama, tanpa menjalankan ulang preprocessing. Cleaning yang digunakan artifact saat ini adalah legacy-v1.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'PRD.md').exists():
    ROOT = ROOT.parent
if not (ROOT / 'PRD.md').exists():
    raise RuntimeError('Buka notebook dari root proyek atau direktori notebooks')
sys.path.insert(0, str(ROOT))
from src.audit import describe_dataset


In [2]:
df = pd.read_csv(ROOT / 'data/processed/processed_reviews.csv')
print(json.dumps(describe_dataset(df), indent=2))


{
  "rows": 49981,
  "columns": [
    "platform",
    "review_id",
    "published_date",
    "score",
    "review_text",
    "clean_text",
    "sentiment"
  ],
  "dtypes": {
    "platform": "str",
    "review_id": "str",
    "published_date": "str",
    "score": "int64",
    "review_text": "str",
    "clean_text": "str",
    "sentiment": "str"
  },
  "missing": {
    "platform": 0,
    "review_id": 0,
    "published_date": 0,
    "score": 0,
    "review_text": 0,
    "clean_text": 0,
    "sentiment": 0
  },
  "duplicate_rows": 0,
  "date_range": [
    "2026-11-19",
    "2026-11-24"
  ],
  "score_counts": {
    "10": 17669,
    "0": 14386,
    "1": 4045,
    "2": 2533,
    "4": 2188,
    "3": 2150,
    "9": 2083,
    "5": 1624,
    "8": 1173,
    "7": 1079,
    "6": 1051
  },
  "platform_counts": {
    "playstation-5": 45580,
    "xbox-series-x": 4401
  },
  "sentiment_counts": {
    "Negative": 25302,
    "Positive": 20925,
    "Neutral": 3754
  },
  "review_id_duplicate_excess": 0,
  

In [3]:
from src.preprocessing import assign_sentiment, clean_text
mismatches = int(df.score.map(assign_sentiment).ne(df.sentiment).sum())
print('Label tidak sesuai rating:', mismatches)
assert mismatches == 0
print('Jumlah kelompok teks bersih dengan label bertentangan:',
      int(df.groupby('clean_text').sentiment.nunique().gt(1).sum()))
sample = df.sample(n=min(200, len(df)), random_state=42)
assert sample.review_text.map(clean_text).eq(sample.clean_text).all()
print('Cleaning legacy-v1 cocok untuk', len(sample), 'sampel deterministik')


Label tidak sesuai rating: 0
Jumlah kelompok teks bersih dengan label bertentangan: 1
Cleaning legacy-v1 cocok untuk 200 sampel deterministik


## Keputusan
Dataset lama dipertahankan. Konflik label tidak diubah secara otomatis. unicode-v2 tersedia untuk eksperimen baru, tetapi belum digunakan pada dataset ini. Jangan mencampurkan unicode-v2 dengan vectorizer legacy-v1. Kebijakan penghapusan review_id lama masih memerlukan klarifikasi sumber.
